<a href="https://colab.research.google.com/github/Manav010203/Quiz/blob/cloudmain/extras/exercises/05_pytorch_going_modular_exercise_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. PyTorch Going Modular Exercises

Welcome to the 05. PyTorch Going Modular exercise template notebook.

There are several questions in this notebook and it's your goal to answer them by writing Python and PyTorch code.

> **Note:** There may be more than one solution to each of the exercises, don't worry too much about the *exact* right answer. Try to write some code that works first and then improve it if you can.

## Resources and solutions

* These exercises/solutions are based on [section 05. PyTorch Going Modular](https://www.learnpytorch.io/05_pytorch_going_modular/) of the Learn PyTorch for Deep Learning course by Zero to Mastery.

**Solutions:**

Try to complete the code below *before* looking at these.

* See a live [walkthrough of the solutions (errors and all) on YouTube](https://youtu.be/ijgFhMK3pp4).
* See an example [solutions notebook for these exercises on GitHub](https://github.com/mrdbourke/pytorch-deep-learning/blob/main/extras/solutions/05_pytorch_going_modular_exercise_solutions.ipynb).

## 1. Turn the code to get the data (from section 1. Get Data) into a Python script, such as `get_data.py`.

* When you run the script using `python get_data.py` it should check if the data already exists and skip downloading if it does.
* If the data download is successful, you should be able to access the `pizza_steak_sushi` images from the `data` directory.

In [1]:
# YOUR CODE HERE
%%writefile get_data.py
import os
import requests
import zipfile
from pathlib import Path

# Setup path to data folder
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

# If the image folder doesn't exist, download it and prepare it...
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

# Download pizza, steak, sushi data
with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
    print("Downloading pizza, steak, sushi data...")
    f.write(request.content)

# Unzip pizza, steak, sushi data
with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
    print("Unzipping pizza, steak, sushi data...")
    zip_ref.extractall(image_path)

# Remove zip file
os.remove(data_path / "pizza_steak_sushi.zip")

Writing get_data.py


In [2]:
# Example running of get_data.py
!python get_data.py

Did not find data/pizza_steak_sushi directory, creating one...
Unzipping pizza, steak, sushi data...


## 2. Use [Python's `argparse` module](https://docs.python.org/3/library/argparse.html) to be able to send the `train.py` custom hyperparameter values for training procedures.
* Add an argument flag for using a different:
  * Training/testing directory
  * Learning rate
  * Batch size
  * Number of epochs to train for
  * Number of hidden units in the TinyVGG model
    * Keep the default values for each of the above arguments as what they already are (as in notebook 05).
* For example, you should be able to run something similar to the following line to train a TinyVGG model with a learning rate of 0.003 and a batch size of 64 for 20 epochs: `python train.py --learning_rate 0.003 batch_size 64 num_epochs 20`.
* **Note:** Since `train.py` leverages the other scripts we created in section 05, such as, `model_builder.py`, `utils.py` and `engine.py`, you'll have to make sure they're available to use too. You can find these in the [`going_modular` folder on the course GitHub](https://github.com/mrdbourke/pytorch-deep-learning/tree/main/going_modular/going_modular).

In [3]:
# YOUR CODE HERE
%%writefile data_setup.py
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()

def create_dataloaders(
    train_dir:str,
    test_dir:str,
    transform: transforms.Compose,
    batch_size:int,
    num_workers:int=NUM_WORKERS):
  train_dataset = datasets.ImageFolder(train_dir, transform=transform)
  test_dataset = datasets.ImageFolder(test_dir, transform=transform)

  class_names = train_dataset.classes

  train_dataloaders = DataLoader(train_dataset,batch_size=batch_size,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True)
  test_dataloaders = DataLoader(test_dataset,batch_size=batch_size,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)

  return train_dataloaders, test_dataloaders, class_names



Writing data_setup.py


In [4]:
%%writefile model_builder.py
import torch
from torch import nn
class TinyVGG(nn.Module):
   def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
      super().__init__()
      self.conv_block_1 = nn.Sequential(
          nn.Conv2d(in_channels=input_shape,
                    out_channels=hidden_units,
                    kernel_size=3,
                    stride=1,
                    padding=0),
          nn.ReLU(),
          nn.Conv2d(in_channels=hidden_units,
                    out_channels=hidden_units,
                    kernel_size=3,
                    stride=1,
                    padding=0),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2,
                        stride=2)
      )
      self.conv_block_2 = nn.Sequential(
          nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
          nn.ReLU(),
          nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
          nn.ReLU(),
          nn.MaxPool2d(2)
      )
      self.classifier = nn.Sequential(
          nn.Flatten(),
           nn.Linear(in_features=hidden_units*13*13,
                    out_features=output_shape)
      )

   def forward(self, x: torch.Tensor):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.classifier(x)
        return x

Writing model_builder.py


In [5]:
%%writefile engine.py
import torch

from tqdm.auto import tqdm
from typing import Dict, List, Tuple

def train_step(model:torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optim: torch.optim.Optimizer,
               device: torch.device) -> Tuple[float,float]:
               model.train()

               train_loss,train_acc =0,0
               for batch,(X,y) in enumerate(dataloader):
                  X,y = X.to(device),y.to(device)
                  y_pred =model(X)
                  loss =loss_fn(y_pred,y)
                  train_loss +=loss.item()
                  optim.zero_grad()
                  loss.backward()
                  optim.step()

                  y_pred_class = torch.argmax(torch.softmax(y_pred,dim=1),dim=1)
                  train_acc +=(y_pred_class==y).sum().item()/len(y_pred)


               train_loss = train_loss/len(dataloader)
               train_acc = train_acc/len(dataloader)
               return train_loss,train_acc

def test_step(model:torch.nn.Module,
              dataloader:torch.utils.data.DataLoader,
              loss_fn:torch.nn.Module,
              device:torch.device) -> Tuple[float,float]:
               model.eval()
               test_loss,test_acc =0,0
               with torch.inference_mode():
                  for batch,(X,y) in enumerate(dataloader):
                     X,y = X.to(device),y.to(device)
                     test_preed_logits = model(X)
                     loss = loss_fn(test_preed_logits,y)
                     test_loss += loss.item()
                     test_pred_labels = test_preed_logits.argmax(dim=1)
                     test_acc += ((test_pred_labels==y).sum().item()/len(test_pred_labels))

               test_loss = test_loss/len(dataloader)
               test_acc = test_acc/len(dataloader)
               return test_loss,test_acc

def train(model:torch.nn.Module,
          train_dataloader:torch.utils.data.DataLoader,
          test_dataloader:torch.utils.data.DataLoader,
          optimizer:torch.optim.Optimizer,
          loss_fn:torch.nn.Module,
          epochs:int,
          device:torch.device)->Dict[str,List]:
                results = {"train_loss":[],
                          "train_acc":[],
                          "test_loss":[],
                          "test_acc":[]}
                for num_epochs in tqdm(range(epochs)):
                   train_loss,train_acc = train_step(model=model,
                                                     dataloader=train_dataloader,
                                                     loss_fn=loss_fn,
                                                     optim=optimizer,
                                                     device=device)
                   test_loss,test_acc = test_step(model=model,
                                                  dataloader=test_dataloader,
                                                  loss_fn=loss_fn,
                                                  device=device)
                   print(f"Epoch: {num_epochs+1} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

                   results["train_loss"].append(train_loss)
                   results["train_acc"].append(train_acc)
                   results["test_loss"].append(test_loss)
                   results["test_acc"].append(test_acc)
                return results

Writing engine.py


In [6]:
%%writefile utils.py

import torch
from pathlib import Path

def save_model(model:torch.nn.Module,
               target_dir:str,
               model_name:str):

  target_dir_path = Path(target_dir)
  target_dir_path.mkdir(parents=True,
                        exist_ok=True)
  assert model_name.endswith(".pth") or model_name.endswith(".pt"), "model_name should end with '.pt' or '.pth'"
  model_save_path = target_dir_path/model_name

  print(f"[INFO] Saving mode to : {model_save_path}")
  torch.save(obj=model.state_dict(),
             f=model_save_path)
#


Writing utils.py


In [29]:
%%writefile train.py
"""
Trains a PyTorch image classification model using device-agnostic code.
"""

import os
import argparse

import torch

from torchvision import transforms

import data_setup, engine, model_builder, utils

# Create a parser
parser = argparse.ArgumentParser(description="Get some hyperparameters.")

# Get an arg for num_epochs
parser.add_argument("--num_epochs",
                     default=10,
                     type=int,
                     help="the number of epochs to train for")

# Get an arg for batch_size
parser.add_argument("--batch_size",
                    default=32,
                    type=int,
                    help="number of samples per batch")

# Get an arg for hidden_units
parser.add_argument("--hidden_units",
                    default=10,
                    type=int,
                    help="number of hidden units in hidden layers")

# Get an arg for learning_rate
parser.add_argument("--learning_rate",
                    default=0.001,
                    type=float,
                    help="learning rate to use for model")

# Create an arg for training directory
parser.add_argument("--train_dir",
                    default="data/pizza_steak_sushi/train",
                    type=str,
                    help="directory file path to training data in standard image classification format")

# Create an arg for test directory
parser.add_argument("--test_dir",
                    default="data/pizza_steak_sushi/test",
                    type=str,
                    help="directory file path to testing data in standard image classification format")

# Get our arguments from the parser
args = parser.parse_args()

# Setup hyperparameters
NUM_EPOCHS = args.num_epochs
BATCH_SIZE = args.batch_size
HIDDEN_UNITS = args.hidden_units
LEARNING_RATE = args.learning_rate
print(f"[INFO] Training a model for {NUM_EPOCHS} epochs with batch size {BATCH_SIZE} using {HIDDEN_UNITS} hidden units and a learning rate of {LEARNING_RATE}")

# Setup directories
train_dir = args.train_dir
test_dir = args.test_dir
print(f"[INFO] Training data file: {train_dir}")
print(f"[INFO] Testing data file: {test_dir}")

# Setup target device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Create transforms
# data_transform = transforms.Compose([
#   transforms.Resize((64, 64)),
#   transforms.RandomHorizontalFlip(p=0.5),
#   transforms.RandomRotation(degrees=15),
#   transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
#   transforms.ToTensor(),
#   transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                        std=[0.229, 0.224, 0.225]),

# ])
data_transform = transforms.Compose([
    transforms.Resize((64,64)),  # Random crop and resize
    transforms.RandomHorizontalFlip(p=0.5),                    # Random horizontal flip
    transforms.RandomVerticalFlip(p=0.2),                      # Random vertical flip (less common)
    transforms.RandomRotation(degrees=20),                     # Random rotation within ±20 degrees
    transforms.ColorJitter(                                     # Strong color augmentation
        brightness=0.3,
        contrast=0.3,
        saturation=0.3,
        hue=0.1
    ),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5), # Random perspective distortion
    transforms.ToTensor(),                                      # Convert image to Tensor
    transforms.Normalize(mean=[0.5, 0.5, 0.5],                 # Normalize: adjust if needed
                         std=[0.5, 0.5, 0.5])
])

# Create DataLoaders with help from data_setup.py
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=data_transform,
    batch_size=BATCH_SIZE
)

# Create model with help from model_builder.py
model = model_builder.TinyVGG(
    input_shape=3,
    hidden_units=HIDDEN_UNITS,
    output_shape=len(class_names)
).to(device)

# Set loss and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),
                             lr=LEARNING_RATE)

# Start training with help from engine.py
engine.train(model=model,
             train_dataloader=train_dataloader,
             test_dataloader=test_dataloader,
             loss_fn=loss_fn,
             optimizer=optimizer,
             epochs=NUM_EPOCHS,
             device=device)

# Save the model with help from utils.py
utils.save_model(model=model,
                 target_dir="models",
                 model_name="05_going_modular_script_mode_tinyvgg_model.pth")

Overwriting train.py


In [33]:

# Example running of train.py
!python train.py --num_epochs 10 --batch_size 128 --hidden_units 128 --learning_rate 0.0003

[INFO] Training a model for 10 epochs with batch size 128 using 128 hidden units and a learning rate of 0.0003
[INFO] Training data file: data/pizza_steak_sushi/train
[INFO] Testing data file: data/pizza_steak_sushi/test
  0% 0/10 [00:00<?, ?it/s]Epoch: 1 | Train Loss: 1.0990 | Train Acc: 0.3251 | Test Loss: 1.0988 | Test Acc: 0.2800
 10% 1/10 [00:01<00:16,  1.81s/it]Epoch: 2 | Train Loss: 1.0793 | Train Acc: 0.3785 | Test Loss: 1.0858 | Test Acc: 0.3733
 20% 2/10 [00:03<00:13,  1.64s/it]Epoch: 3 | Train Loss: 1.0207 | Train Acc: 0.5349 | Test Loss: 1.0220 | Test Acc: 0.5200
 30% 3/10 [00:04<00:11,  1.59s/it]Epoch: 4 | Train Loss: 0.9667 | Train Acc: 0.5518 | Test Loss: 1.0238 | Test Acc: 0.4000
 40% 4/10 [00:06<00:10,  1.80s/it]Epoch: 5 | Train Loss: 0.9080 | Train Acc: 0.6035 | Test Loss: 1.0393 | Test Acc: 0.4267
 50% 5/10 [00:08<00:09,  1.86s/it]Epoch: 6 | Train Loss: 0.8874 | Train Acc: 0.5637 | Test Loss: 1.0065 | Test Acc: 0.5467
 60% 6/10 [00:10<00:06,  1.74s/it]Epoch: 7 | Trai

## 3. Create a Python script to predict (such as `predict.py`) on a target image given a file path with a saved model.

* For example, you should be able to run the command `python predict.py some_image.jpeg` and have a trained PyTorch model predict on the image and return its prediction.
* To see example prediction code, check out the [predicting on a custom image section in notebook 04](https://www.learnpytorch.io/04_pytorch_custom_datasets/#113-putting-custom-image-prediction-together-building-a-function).
* You may also have to write code to load in a trained model.

In [34]:
# YOUR CODE HERE
%%writefile predict.py
import torch
import torchvision
import argparse

import model_builder

parser = argparse.ArgumentParser()

parser.add_argument("--image",
                    help="target image filepath to predict on")

parser.add_argument("--model_path",
                    default="models/05_going_modular_script_mode_tinyvgg_model.pth",
                    type=str,
                    help="target model to use for prediction filepath")

args = parser.parse_args()

class_names = ["pizza","steak","sushi"]

device ="cuda" if torch.cuda.is_available() else "cpu"

IMG_PATH = args.image
print(f"[INFO] Predicting on {IMG_PATH}")

def load_model(filepath=args.model_path):
  model = model_builder.TinyVGG(input_shape=3,
                                hidden_units=128,
                                output_shape=3).to(device)
  print(f"[INFO] Loading in model from: {filepath}")
  model.load_state_dict(torch.load(filepath))
  return model

def predict_on_image(image_path=IMG_PATH,
                     filepath=args.model_path):
  model = load_model(filepath)
  image = torchvision.io.read_image(str(IMG_PATH)).type(torch.float32)
  image = image/255.
  transform = torchvision.transforms.Resize(size=(64,64))
  image = transform(image)

  model.eval()
  with torch.inference_mode():
    image=image.to(device)
    image = image.unsqueeze(dim=0)
    pred_logits = model(image)
    pred_probs = torch.softmax(pred_logits,dim=1)
    pred_label = torch.argmax(pred_probs,dim=1)
    pred_class = class_names[pred_label]
  print(f"[INFO] Pred class: {pred_class}, Pred Prob: {pred_probs.max():.3f}")

if __name__ == "__main__":
  predict_on_image()

Overwriting predict.py


In [35]:
# Example running of predict.py
!python predict.py --image data/pizza_steak_sushi/test/sushi/175783.jpg

[INFO] Predicting on data/pizza_steak_sushi/test/sushi/175783.jpg
[INFO] Loading in model from: models/05_going_modular_script_mode_tinyvgg_model.pth
[INFO] Pred class: steak, Pred Prob: 0.482
